In [313]:
import numpy as np
from pytreenet.operators import Hamiltonian, TensorProduct
from pytreenet.ttno.state_diagram import TTNOFinder
from pytreenet.ttno.ttno_class import TreeTensorNetworkOperator
from pytreenet.ttns import TreeTensorNetworkState
from fractions import Fraction
from TreeTopologies import random_impurity_binary_tree_ttn
from pytreenet.time_evolution.bug import BUGConfig, BUG
from pytreenet.time_evolution.time_evolution import TimeEvoMode
from Hamiltonians import SIAM_TTNO

Parameters. Same parameters for up and down spin.

In [314]:
angle = -np.pi/2
eshift = 0
n_bath = 4
chi_max = 16
U = 1.5
dt = 0.1
tend = 10

# hopping terms
hop = np.ones(n_bath)

# bath on-site energy
eb = np.ones(n_bath)

# impurity on-site energy
ei = -U / 2

binary_ttn = random_impurity_binary_tree_ttn(n_bath=n_bath, chi_max=chi_max)
state = TreeTensorNetworkState.from_ttn(binary_ttn)
state.normalise()

np.float64(116671.18876268393)

The SIAM Hamiltonian:
$$\hat H = U\,\hat n_{0\uparrow}\hat n_{0\downarrow} -\mu \sum_{\sigma} \hat C^{\dagger}_{0\sigma}\hat C_{0\sigma} + \sum_{i=1}^{N}\sum_{\sigma} \varepsilon_i\,
        \hat C^{\dagger}_{i\sigma}\hat C_{i\sigma} + t_i\,\hat C^{\dagger}_{0\sigma}\hat C_{i\sigma}
      + t_i^{*}\,\hat C^{\dagger}_{i\sigma}\hat C_{0\sigma}.$$

The site interaction are represented by lines and either represent hopping terms or for the impurity site a coulomb-like interaction.

<img src="./images/SIAM.png" width="500">

To reduce the order of the tensors, we introduce auxiliary tensors that do not represent physical sites.

<img src="./images/ttn_binary_tree.png" width="500">

Create empty hamiltonian to which we gonna add each term of the hamiltonian

In [315]:
hamiltonian = Hamiltonian()

Setting up the operators, the pauli-z we need to implement fermionic (orthogonal) anticommutation. 
The identities are required since they will act on the sites (physical and non-physical), on which no operator acts.
Or in different words the identities are used to fill the operator string. 
The non-physical tensors have a physical leg of dimension 1.

In [316]:
c = np.array([[0, 1], [0, 0]])
n = np.array([[0, 0], [0, 1]])
pz = np.array([[1, 0], [0, -1]])
conversion_dict = {}
conversion_dict["I2"] = np.eye(2)
conversion_dict["I1"] = np.eye(1)
conversion_dict["c"] = c
conversion_dict["c*"] = c.T
conversion_dict["n"] = n
conversion_dict["pz"] = pz

I just keep frac at 1. The coefficient map will be used to store the coefficient of each term.

In [317]:
frac = Fraction(1)
coeff_map = dict()

For time evolution at an angle, one can just multiply the hamiltonian with an angle-factor.
ATTENTION: angle between 0 and pi/2. One develops the state backwards in imaginary time -> High energy states are amplified.

$$\exp(-i(\cos(\phi)+i\sin(\phi))\hat Ht)$$

In [318]:
angle_fact = np.cos(angle) + 1j * np.sin(angle)
angle_fact = np.real_if_close(angle_fact)
angle_fact = -1j*(np.real_if_close(1j*(angle_fact)))

Include energy shift term, typically by the groundstate energy.
First term we are gonna iclude but not part of the Hamiltonian.

$$\exp(-i(\hat H - E_{0})t)$$

In [319]:
e_shift_term = TensorProduct()
e_shift_coeff_id = "E"
coeff_map[e_shift_coeff_id] = eshift * angle_fact
hamiltonian.add_term((frac, e_shift_coeff_id, e_shift_term))

Construct all terms including an operator acting on a bath site.
The Assumed order for the Jordan-Wigner Transformation is:
bath0(up)|...|bath{Nb-1}(up)|imp(up)|imp(down)|bath0(down)|...|bath{Nb-1}(down)

ATTENTION: The naming conventions of the Hamiltonian terms and the ttn-nodes must match.

In [320]:
for i in range(n_bath):

        pzs_up = {f"bath{j}(up)": "pz" for j in range(i + 1, n_bath)}

        # Hopping term from impurity-site to bath-site i for up-spin.
        hop_term_up = TensorProduct(
            {f"bath{i}(up)": "c*"} | pzs_up | {"imp(up)": "c"}
        )
        hop_coeff_id = f"t{i}*"
        coeff_map[hop_coeff_id] = hop[i] * angle_fact
        hamiltonian.add_term((frac, hop_coeff_id, hop_term_up))

        # Hopping term bath-site i to impurity-site for up-spin.
        hop_term_up_dag = TensorProduct(
            {f"bath{i}(up)": "c"} | pzs_up | {"imp(up)": "c*"}
        )
        hop_conj_coeff_id = f"t{i}"
        coeff_map[hop_conj_coeff_id] = np.conj(hop[i]) * angle_fact
        hamiltonian.add_term((frac, hop_conj_coeff_id, hop_term_up_dag))

        # On-site energy term for bath-site i.
        os_term_up = TensorProduct({f"bath{i}(up)": "n"})
        os_coeff_id = f"e{i}"
        coeff_map[os_coeff_id] = eb[i] * angle_fact
        hamiltonian.add_term((frac, os_coeff_id, os_term_up))

        ######
        # Same terms, but for spin down. 
        # Jordan-Wigner order differs though.
        pzs_down = {f"bath{j}(down)": "pz" for j in range(0, i)}


        hop_term_down = TensorProduct(
            {"imp(down)": "c"} | pzs_down | {f"bath{i}(down)": "c*"}
        )
        hamiltonian.add_term((frac, hop_coeff_id, hop_term_down))


        hop_term_down_dag = TensorProduct(
            {"imp(down)": "c*"} | pzs_down | {f"bath{i}(down)": "c"}
        )
        hamiltonian.add_term((frac, hop_conj_coeff_id, hop_term_down_dag))


        os_term_down = TensorProduct({f"bath{i}(down)": "n"})
        hamiltonian.add_term((frac, os_coeff_id, os_term_down))

On-site energy terms for the Impurity site

In [321]:
# both terms have the same coefficient (chemical potential)
os_imp_coeff_id = "eimp"
coeff_map[os_imp_coeff_id] = ei * angle_fact

# up-spin
os_term_up = TensorProduct({"imp(up)": "n"})
hamiltonian.add_term((frac, os_imp_coeff_id, os_term_up))

# down-spin
os_term_down = TensorProduct({"imp(down)": "n"})
hamiltonian.add_term((frac, os_imp_coeff_id, os_term_down))

Interaction-term

In [322]:
inter_term = TensorProduct({"imp(up)": "n", "imp(down)": "n"})
inter_coeff_id = "U"
coeff_map["U"] = U * angle_fact
hamiltonian.add_term((frac, inter_coeff_id, inter_term))

Creating the TTNO, the reference_tree gives the TTN structure, here two connected binary trees.

In [323]:
hamiltonian.conversion_dictionary = conversion_dict
hamiltonian.coeffs_mapping = coeff_map

siam_ttno = TreeTensorNetworkOperator.from_hamiltonian(
    hamiltonian=hamiltonian, reference_tree=state, method=TTNOFinder.SGE
)

Once you have the Hamiltonian and the underlying state. (Here the state is just a ttn with randomly generated tensors respecting a certain maximal bond dimension chi_max. Would usually be a result of a DMRG calclation for example.)
Time evolution with pytreenet is straightforward.

In [ ]:
raw_ham = SIAM_TTNO(Nb=n_bath, hop=hop, eb=eb, ei=ei, U=U, state=state)

config = BUGConfig(max_bond_dim=chi_max, time_evo_mode=TimeEvoMode.CHEBYSHEV)

bug = BUG(initial_state=state, hamiltonian=siam_ttno,
                       time_step_size=dt, final_time=tend, operators={},
                       config=config)

for i in range(int(tend//dt)):
    bug.run_one_time_step()
    bug.state.normalise()
    E = bug.state.ttno_expectation_value(raw_ham)
    print("energy: ", E)


energy:  (2.4814928927595163-3.885780586188048e-16j)
energy:  (1.621212143491091+1.1102230246251565e-16j)
energy:  (0.9101036160114383-1.5265566588595902e-16j)
energy:  (0.3453752402675213-2.0816681711721685e-17j)
energy:  (-0.09495176008665406+4.163336342344337e-17j)
energy:  (-0.43994587876919455-1.0408340855860843e-16j)
energy:  (-0.7171854041947519-5.828670879282072e-16j)
energy:  (-0.9488368886637482-2.7755575615628914e-16j)
energy:  (-1.151113087859971+4.163336342344337e-16j)
energy:  (-1.3351986738862953+1.1102230246251565e-16j)
energy:  (-1.5084394641756118+2.220446049250313e-16j)
energy:  (-1.6753230858792953+3.885780586188048e-16j)
energy:  (-1.838185529526196+2.7755575615628914e-17j)
energy:  (-1.9977216148421202+6.38378239159465e-16j)
energy:  (-2.1534020815239554+3.0531133177191805e-16j)
energy:  (-2.303870877520918-1.942890293094024e-16j)
energy:  (-2.4473458904223166+6.106226635438361e-16j)
energy:  (-2.5819998497610905-2.220446049250313e-16j)
energy:  (-2.7062744639559-